<a href="https://colab.research.google.com/github/vivekgautamgv/Python-For-Finance/blob/main/Intern_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install numpy

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/Football Data Test Task(Raw Data).csv')

# Initialize new columns for aggregated statistics
stats = ['FTHG', 'FTAG', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']
for stat in stats:
    for period in [5, 15, 38]:
        df[f'{stat}_L{period}'] = 0

# Function to calculate statistics for a given team
def calculate_team_stats(team, df, period):
    # Filter past matches for the team
    past_matches = df[(df['HomeTeam'] == team) | (df['AwayTeam'] == team)].tail(period)

    # Initialize statistics
    stats = {
        'FTHG': 0, 'FTAG': 0, 'HS': 0, 'AS': 0, 'HST': 0, 'AST': 0,
        'HF': 0, 'AF': 0, 'HC': 0, 'AC': 0, 'HY': 0, 'AY': 0, 'HR': 0, 'AR': 0
    }

    # Calculate statistics
    for _, match in past_matches.iterrows():
        if match['HomeTeam'] == team:
            stats['FTHG'] += match['FTHG']
            stats['HS'] += match['HS']
            stats['HST'] += match['HST']
            stats['HF'] += match['HF']
            stats['HC'] += match['HC']
            stats['HY'] += match['HY']
            stats['HR'] += match['HR']
        if match['AwayTeam'] == team:
            stats['FTAG'] += match['FTAG']
            stats['AS'] += match['AS']
            stats['AST'] += match['AST']
            stats['AF'] += match['AF']
            stats['AC'] += match['AC']
            stats['AY'] += match['AY']
            stats['AR'] += match['AR']

    return stats

# Iterate over each match to calculate statistics
for index, row in df.iterrows():
    home_team = row['HomeTeam']
    away_team = row['AwayTeam']

    for period in [5, 15, 38]:
        # Calculate stats for home team
        home_stats = calculate_team_stats(home_team, df.iloc[:index], period)
        for stat in stats:
            df.at[index, f'{stat}_L{period}'] = home_stats[stat]

        # Calculate stats for away team
        away_stats = calculate_team_stats(away_team, df.iloc[:index], period)
        for stat in stats:
            df.at[index, f'{stat}_L{period}'] = away_stats[stat]

# Save the updated DataFrame to a new CSV file
df.to_csv('football_dataset_with_aggregated_stats.csv', index=False)


In [3]:
import pandas as pd

# Load the updated CSV file
df_updated = pd.read_csv('football_dataset_with_aggregated_stats.csv')

# Display the column names
print("Column Names:")
print(df_updated.columns.tolist())

# Display the first few rows of the DataFrame
print("\nHead of the DataFrame:")
print(df_updated.head())


Column Names:
['Incremental_ID', 'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'FTHG_L5', 'FTHG_L15', 'FTHG_L38', 'FTAG_L5', 'FTAG_L15', 'FTAG_L38', 'HS_L5', 'HS_L15', 'HS_L38', 'AS_L5', 'AS_L15', 'AS_L38', 'HST_L5', 'HST_L15', 'HST_L38', 'AST_L5', 'AST_L15', 'AST_L38', 'HF_L5', 'HF_L15', 'HF_L38', 'AF_L5', 'AF_L15', 'AF_L38', 'HC_L5', 'HC_L15', 'HC_L38', 'AC_L5', 'AC_L15', 'AC_L38', 'HY_L5', 'HY_L15', 'HY_L38', 'AY_L5', 'AY_L15', 'AY_L38', 'HR_L5', 'HR_L15', 'HR_L38', 'AR_L5', 'AR_L15', 'AR_L38']

Head of the DataFrame:
   Incremental_ID Div       Date Time       HomeTeam    AwayTeam  FTHG  FTAG  \
0               1  E0  8/13/2005  NaN    Aston Villa      Bolton     2     2   
1               2  E0  8/13/2005  NaN        Everton  Man United     0     2   
2               3  E0  8/13/2005  NaN         Fulham  Birmingham     0     0   
3               4  E0

In [4]:
import pandas as pd
import numpy as np
from datetime import datetime

class FootballDataProcessor:
    def __init__(self):
        self.raw_data = None
        self.processed_data = None

    def load_data(self, csv_path):
        """Load and prepare the raw football data."""
        self.raw_data = pd.read_csv('/content/Football Data Test Task(Raw Data).csv')
        # Convert date to datetime
        self.raw_data['Date'] = pd.to_datetime(self.raw_data['Date'])
        # Sort by date to ensure chronological processing
        self.raw_data = self.raw_data.sort_values('Date')

    def get_team_history(self, team, current_date, last_n_matches):
        """Get the last N matches for a specific team before the given date."""
        mask = (
            ((self.raw_data['HomeTeam'] == team) |
             (self.raw_data['AwayTeam'] == team)) &
            (self.raw_data['Date'] < current_date)
        )
        team_matches = self.raw_data[mask].tail(last_n_matches)

        return team_matches

    def calculate_team_stats(self, team, current_date, last_n_matches):
        """Calculate statistics for a team based on their last N matches."""
        history = self.get_team_history(team, current_date, last_n_matches)

        stats = {
            'FTHG': 0, 'FTAG': 0,  # Goals
            'HS': 0, 'AS': 0,      # Shots
            'HST': 0, 'AST': 0,    # Shots on Target
            'HF': 0, 'AF': 0,      # Fouls
            'HC': 0, 'AC': 0,      # Corners
            'HY': 0, 'AY': 0,      # Yellow Cards
            'HR': 0, 'AR': 0,      # Red Cards
            'HW': 0, 'HL': 0, 'HD': 0,  # Home results
            'AW': 0, 'AL': 0, 'AD': 0   # Away results
        }

        if len(history) == 0:
            return {k: 0 for k in stats.keys()}

        for _, match in history.iterrows():
            if match['HomeTeam'] == team:
                # When team played at home
                stats['FTHG'] += match['FTHG']
                stats['HS'] += match['HS']
                stats['HST'] += match['HST']
                stats['HF'] += match['HF']
                stats['HC'] += match['HC']
                stats['HY'] += match['HY']
                stats['HR'] += match['HR']

                # Count results
                if match['FTR'] == 'H':
                    stats['HW'] += 1
                elif match['FTR'] == 'A':
                    stats['HL'] += 1
                else:
                    stats['HD'] += 1

            else:
                # When team played away
                stats['FTAG'] += match['FTAG']
                stats['AS'] += match['AS']
                stats['AST'] += match['AST']
                stats['AF'] += match['AF']
                stats['AC'] += match['AC']
                stats['AY'] += match['AY']
                stats['AR'] += match['AR']

                # Count results
                if match['FTR'] == 'A':
                    stats['AW'] += 1
                elif match['FTR'] == 'H':
                    stats['AL'] += 1
                else:
                    stats['AD'] += 1

        return stats

    def process_data(self):
        """Process the entire dataset to include historical statistics."""
        # Create a copy of raw data for processing
        self.processed_data = self.raw_data.copy()

        # Initialize columns for historical stats
        periods = [5, 15, 38]
        stats_columns = [
            'FTHG', 'FTAG', 'HS', 'AS', 'HST', 'AST',
            'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR',
            'HW', 'HL', 'HD', 'AW', 'AL', 'AD'
        ]

        for period in periods:
            for stat in stats_columns:
                self.processed_data[f'{stat}_L{period}'] = 0

        # Process each match
        for idx, match in self.processed_data.iterrows():
            # Calculate home team stats
            home_stats_5 = self.calculate_team_stats(match['HomeTeam'], match['Date'], 5)
            home_stats_15 = self.calculate_team_stats(match['HomeTeam'], match['Date'], 15)
            home_stats_38 = self.calculate_team_stats(match['HomeTeam'], match['Date'], 38)

            # Calculate away team stats
            away_stats_5 = self.calculate_team_stats(match['AwayTeam'], match['Date'], 5)
            away_stats_15 = self.calculate_team_stats(match['AwayTeam'], match['Date'], 15)
            away_stats_38 = self.calculate_team_stats(match['AwayTeam'], match['Date'], 38)

            # Update processed data with calculated stats
            for stat in stats_columns:
                self.processed_data.at[idx, f'{stat}_L5'] = home_stats_5[stat] + away_stats_5[stat]
                self.processed_data.at[idx, f'{stat}_L15'] = home_stats_15[stat] + away_stats_15[stat]
                self.processed_data.at[idx, f'{stat}_L38'] = home_stats_38[stat] + away_stats_38[stat]

    def save_processed_data(self, output_path):
        """Save the processed data to a CSV file."""
        if self.processed_data is not None:
            self.processed_data.to_csv(output_path, index=False)
        else:
            raise ValueError("No processed data available. Run process_data() first.")

# Usage example
def main():
    # Initialize the processor
    processor = FootballDataProcessor()

    # Load the data
    processor.load_data('football_data.csv')

    # Process the data
    processor.process_data()

    # Save the processed data
    processor.save_processed_data('processed_football_data.csv')

if __name__ == "__main__":
    main()

In [7]:
import pandas as pd

# Load the updated CSV file
df_updated = pd.read_csv('/content/processed_football_data.csv')

# Display the column names
print("Column Names:")
print(df_updated.columns.tolist())

# Display the first few rows of the DataFrame
print("\nHead of the DataFrame:")
print(df_updated.head())


Column Names:
['Incremental_ID', 'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'FTHG_L5', 'FTAG_L5', 'HS_L5', 'AS_L5', 'HST_L5', 'AST_L5', 'HF_L5', 'AF_L5', 'HC_L5', 'AC_L5', 'HY_L5', 'AY_L5', 'HR_L5', 'AR_L5', 'HW_L5', 'HL_L5', 'HD_L5', 'AW_L5', 'AL_L5', 'AD_L5', 'FTHG_L15', 'FTAG_L15', 'HS_L15', 'AS_L15', 'HST_L15', 'AST_L15', 'HF_L15', 'AF_L15', 'HC_L15', 'AC_L15', 'HY_L15', 'AY_L15', 'HR_L15', 'AR_L15', 'HW_L15', 'HL_L15', 'HD_L15', 'AW_L15', 'AL_L15', 'AD_L15', 'FTHG_L38', 'FTAG_L38', 'HS_L38', 'AS_L38', 'HST_L38', 'AST_L38', 'HF_L38', 'AF_L38', 'HC_L38', 'AC_L38', 'HY_L38', 'AY_L38', 'HR_L38', 'AR_L38', 'HW_L38', 'HL_L38', 'HD_L38', 'AW_L38', 'AL_L38', 'AD_L38']

Head of the DataFrame:
   Incremental_ID Div        Date Time       HomeTeam    AwayTeam  FTHG  FTAG  \
0               1  E0  2005-08-13  NaN    Aston Villa      Bolton     2     2   
1    